# Local loopback

Literal loopback targets and a server-issued nonce; closed refusals, then a real local HTTP round trip whose server and connections are closed.

Open this file in JupyterLab or VS Code with the **Deno** kernel. From a repository checkout, install Deno 2.5.6, Python 3.13.9 and uv 0.9.9 (the `.tool-versions` pins), run `uv sync --frozen` and `uv run --frozen deno jupyter --install`, then select Deno and Run All. Any Jupyter-capable editor can use that installed kernel. See [launch and automated checks](README.md).

The saved notebook installs exact published packages. `pnpm notebooks:local` checks the packed candidate; `pnpm notebooks:published` runs these cells unchanged against npm. Fixed times describe a demonstration, not current credentials. Keys and signatures are generated, never golden output. Host authorization, replay and revocation stay outside this notebook.


In [1]:
import signerMetadata from "npm:@bounded-authority-protocol/signer@2.0.0/package.json" with { type: "json" };
import verifierMetadata from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
if (signerMetadata.version !== "2.0.0" || verifierMetadata.version !== "2.0.0") throw new Error("package version mismatch");
console.log("exact signer and verifier releases loaded");

exact signer and verifier releases loaded


In [2]:
import { Buffer } from "node:buffer";
import { generateKeyPairSync, sign } from "node:crypto";
import { jwkFromPublicKey, thumbprint, v3 } from "npm:@bounded-authority-protocol/verifier@2.0.0";
import type { Es256KeyHandle, KeyHandle, KeyRole } from "npm:@bounded-authority-protocol/signer@2.0.0";

const fromBase64url = (text: string): Uint8Array => new Uint8Array(Buffer.from(text, "base64url"));

/** An Ed25519 handle over a fresh in-process key (contract-majors 1 and 2). */
function testKeyHandle(keyId: string, role: KeyRole = "holder"): KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ed25519");
  const raw = fromBase64url(publicKey.export({ format: "jwk" }).x!);
  return {
    sign: (message) => new Uint8Array(sign(null, message, privateKey)),
    publicKey: () => raw,
    thumbprint: () => thumbprint(jwkFromPublicKey(raw)),
    keyIdentity: () => ({ keyId, publicKey: raw }),
    signingIdentity: () => ({ role, keyId, publicKey: raw }),
  };
}

/** An ES256 handle over a fresh in-process P-256 key (contract-major 3). */
function testEs256KeyHandle(keyId: string, role: KeyRole = "holder"): Es256KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ec", { namedCurve: "P-256" });
  const { x, y } = publicKey.export({ format: "jwk" }) as { x: string; y: string };
  const raw = new Uint8Array([0x04, ...fromBase64url(x), ...fromBase64url(y)]);
  return {
    sign: (message) => new Uint8Array(sign("sha256", message, { key: privateKey, dsaEncoding: "ieee-p1363" })),
    publicKey: () => raw,
    thumbprint: () => v3.thumbprint({ crv: "P-256", kty: "EC", x, y }),
    keyIdentity: () => ({ keyId, publicKey: raw }),
    signingIdentity: () => ({ role, keyId, publicKey: raw }),
  };
}
console.log("demonstration key constructors ready");

demonstration key constructors ready


In [3]:
import { signGrant, signLocalLoopbackReport } from "npm:@bounded-authority-protocol/signer@2.0.0";
import { checkLocalLoopbackHttpEnvelope, strUtf8, type Tagged } from "npm:@bounded-authority-protocol/verifier@2.0.0";


const issuer = testKeyHandle("issuer-key-1", "issuer");
const helper = testKeyHandle("helper-key-1"); // an agent on the same machine

const grant = await signGrant({
  issuer: "https://issuer.example", grantId: "urn:example:grant:notes",
  audiences: ["https://notes.example"],
  issuedAt: 1_767_225_600, notBefore: 1_767_225_600, expiresAt: 1_767_229_200,
  holderThumbprint: await helper.thumbprint(),
  operations: [{ name: "open_note", selectors: [{ kind: "all" }] }],
}, issuer);
if (!grant.ok) throw new Error(`signGrant: ${grant.error}`);

const nonce = "c8f2a1"; // issued by the local app for this request
const request = {
  method: "POST", targetUri: "http://127.0.0.1:47321/rpc",
  invocationId: "9c4b2e7a-1d3f-4c5b-a6e7-8f9a0b1c2d3e", operation: "open_note",
  castArguments: { t: "object", v: new Map<string, Tagged>([["title", { t: "string", v: strUtf8("Groceries") }]]) } satisfies Tagged,
};
const envelope = await signLocalLoopbackReport({ grantCompact: grant.value.grant, ...request, nonce }, helper, { issuedAt: 1_767_225_660 });
if (!envelope.ok) throw new Error(`signLocalLoopbackReport: ${envelope.error}`);

const facts = checkLocalLoopbackHttpEnvelope(envelope.value.grant, envelope.value.proof, {
  ...request,
  trustedIssuer: { keyId: "issuer-key-1", publicKey: await issuer.publicKey() },
  issuer: "https://issuer.example", audience: "https://notes.example",
  evaluationTime: 1_767_225_700, clockSkew: 60, proofMaxAge: 300,
  nonce: { kind: "required", value: nonce },
});
console.log(`loopback envelope verifies: ${facts.ok}`);
if (!facts.ok) throw new Error("loopback verification");
const missingNonce = await signLocalLoopbackReport({ ...request, grantCompact: grant.value.grant }, helper, { issuedAt: 1_767_225_660 });
const hostname = await signLocalLoopbackReport({ ...request, targetUri: "http://localhost:47321/rpc", nonce, grantCompact: grant.value.grant }, helper, { issuedAt: 1_767_225_660 });
if (missingNonce.ok || missingNonce.error !== "invalid_report" || hostname.ok || hostname.error !== "producer_error") throw new Error("loopback refusal");

loopback envelope verifies: true


In [4]:
import { createServer } from "node:http";
import { once } from "node:events";
import { signGrant, signLocalLoopbackReport } from "npm:@bounded-authority-protocol/signer@2.0.0";
import { checkLocalLoopbackHttpEnvelope, strUtf8 } from "npm:@bounded-authority-protocol/verifier@2.0.0";

{
const issuer = testKeyHandle("issuer-key", "issuer"), holder = testKeyHandle("holder-key");
const grant = await signGrant({ issuer: "https://issuer.example", grantId: "urn:example:grant:http",
  audiences: ["https://resource.example"], issuedAt: 1000, notBefore: 1000, expiresAt: 2000,
  holderThumbprint: await holder.thumbprint(), operations: [{ name: "inspect", selectors: [{ kind: "all" }] }] }, issuer);
if (!grant.ok) throw new Error(grant.error);
const publicKey = await issuer.publicKey();
const nonce = "server-challenge", invocationId = "550e8400-e29b-41d4-a716-446655440000";
const server = createServer((req, res) => {
  const proof = String(req.headers["x-example-proof"] ?? "");
  const receivedGrant = String(req.headers["x-example-grant"] ?? "");
  const result = checkLocalLoopbackHttpEnvelope(strUtf8(receivedGrant), strUtf8(proof), {
    method: req.method!, targetUri: `http://127.0.0.1:${port}${req.url}`, invocationId,
    operation: "inspect", castArguments: { t: "null" },
    trustedIssuer: { keyId: "issuer-key", publicKey }, issuer: "https://issuer.example", audience: "https://resource.example",
    evaluationTime: 1500, clockSkew: 0, proofMaxAge: 300, nonce: { kind: "required", value: nonce },
  });
  res.writeHead(result.ok ? 200 : 401, { "content-type": "application/json" });
  res.end(JSON.stringify({ verified: result.ok }));
});
server.listen(0, "127.0.0.1");
await once(server, "listening");
const address = server.address();
if (!address || typeof address === "string") throw new Error("listener address");
const port = address.port;
try {
  const targetUri = `http://127.0.0.1:${port}/inspect`;
  const proof = await signLocalLoopbackReport({ grantCompact: grant.value.grant, operation: "inspect",
    method: "POST", targetUri, invocationId, castArguments: { t: "null" }, nonce }, holder,
    { issuedAt: 1500, proofId: "urn:example:proof:http" });
  if (!proof.ok) throw new Error(proof.error);
  const headers = { "x-example-grant": new TextDecoder().decode(proof.value.grant), "x-example-proof": new TextDecoder().decode(proof.value.proof) };
  const success = await fetch(targetUri, { method: "POST", headers });
  if (success.status !== 200 || !(await success.json() as { verified: boolean }).verified) throw new Error("HTTP verification failed");
  const refusal = await fetch(targetUri + "-changed", { method: "POST", headers });
  if (refusal.status !== 401 || (await refusal.json() as { verified: boolean }).verified) throw new Error("changed URI accepted");
  console.log("HTTP: exact request verifies; changed target refused");
} finally {
  server.closeAllConnections();
  await new Promise<void>((resolve, reject) => server.close((error) => error ? reject(error) : resolve()));
}
}

HTTP: exact request verifies; changed target refused


## Cleanup

These cells keep keys and compacts only in the kernel. Shut down the kernel to discard them. No private material is persisted. The runner shuts down the kernel and removes its consumer directory, package caches and temporary kernel registration, and fails on files left in the notebook working directory.

Continue with the [guides](../guides/getting-started.md).
